In [ ]:

import json
import re
import random
import time
from pathlib import Path
from tqdm.notebook import tqdm
from mlx_lm import load, generate
from collections import Counter


# Cell 11: Load parsed emails from cache

cache_path = PROJECT / "data/parsed/emails.json"

with open(cache_path, 'r', encoding='utf-8') as f:
    parsed_emails = json.load(f)

print(f"✅ Loaded {len(parsed_emails):,} emails from cache")

# Cell 12: Random sampling
random.seed(42)

# Pick 500 random emails
sample_size = 500
sample_emails = random.sample(parsed_emails,sample_size)

print(f"Total emails: {len(parsed_emails):,}")
print(f"Sample size: {len(sample_emails)}")

# Preview one sample
print(f"\n=== SAMPLE EMAIL #1 ===")
print(f"Subject: {sample_emails[0]['subject']}")
print(f"Sender: {sample_emails[0]['sender']}")
print(f"Body: {sample_emails[0]['body'][:300]}...")

# Cell 13: Classification prompt template

CLASSIFICATION_PROMPT = """You are an email classifier. Analyze this email and categorize it.

EMAIL:
Subject: {subject}
From: {sender}
Body: {body}

TASK:
Classify this email into exactly ONE category.

CATEGORIES:
- finance: Banks, payments, transactions, investments, credit cards, loans, UPI, wallets
- shopping: Orders, deliveries, purchases, e-commerce
- social: Social networks, personal messages, invitations
- work: Job-related, recruitment, office, meetings, projects
- newsletter: Digests, subscriptions, blogs, articles
- promotional: Marketing, offers, discounts, advertisements
- other: Anything that doesn't fit above

OUTPUT FORMAT (JSON only, no other text):
{{"category": "<category>", "confidence": "<high/medium/low>", "reason": "<brief 5-10 word reason>"}}
"""

def build_prompt(email_data):
    """Build classification prompt for one email."""
    return CLASSIFICATION_PROMPT.format(
        subject=email_data['subject'][:200],
        sender=email_data['sender'][:100],
        body=email_data['body'][:2000]
    )

# Test: See what prompt looks like
test_prompt = build_prompt(sample_emails[0])
print(f"Prompt length: {len(test_prompt)} characters")
print(f"\n=== PROMPT PREVIEW ===\n{test_prompt[:1000]}...")

# Cell 14: Load Phi-3 model
model_path = str(PROJECT / "models/base/phi3-mini")

print("Loading Phi-3 model...")
model, tokenizer = load(model_path)
print("✅ Model loaded")

# Cell 15: Test classification on one email
test_email = sample_emails[0]

# Build prompt
prompt = build_prompt(test_email)

# Send to Phi-3
print("Classifying email...")
print(f"Subject: {test_email['subject'][:80]}...")
print("-" * 50)

response = generate(
    model, 
    tokenizer, 
    prompt=prompt,
    max_tokens=100,
    verbose=False
)

print(f"\n=== PHI-3 RESPONSE ===\n{response}")

# Cell 16: JSON extraction helper

def extract_json(response):
    """Extract JSON object from LLM response."""

    # Find JSON pattern in response
    match = re.search(r'\{[^{}]*\}', response)

    if(match):
          try:
              return json.loads(match.group())
          except json.JSONDecodeError:
              return None
    return None

# Test on previous response
parsed = extract_json(response)

print("=== EXTRACTED JSON ===")
print(parsed)
print(f"\nCategory: {parsed['category']}")
print(f"Confidence: {parsed['confidence']}")
print(f"Reason: {parsed['reason']}")

# Cell 17: Classify all sample emails
results = []
failed = 0

print(f"Classifying {len(sample_emails)} emails...")
print("Estimated time: ~5 minutes\n")

start_time = time.time()

for i, email_data in enumerate(tqdm(sample_emails, desc="Classifying")):
    try:
        # Build prompt
        prompt = build_prompt(email_data)
        
        # Get classification
        response = generate(
            model, 
            tokenizer, 
            prompt=prompt,
            max_tokens=100,
            verbose=False
        )
        
        # Extract JSON
        parsed = extract_json(response)
        
        if parsed:
            results.append({
                'id': email_data.get('id', i),
                'subject': email_data['subject'],
                'sender': email_data['sender'],
                'category': parsed.get('category', 'other'),
                'confidence': parsed.get('confidence', 'low'),
                'reason': parsed.get('reason', '')
            })
        else:
            failed += 1
            
    except Exception as e:
        failed += 1
        continue

elapsed = time.time() - start_time

print(f"\n✅ Classified: {len(results)}")
print(f"❌ Failed: {failed}")
print(f"⏱️ Time: {elapsed/60:.1f} minutes")
print(f"⚡ Speed: {len(results)/elapsed:.1f} emails/sec")

# Cell 18: Category distribution

categories = Counter([r['category'] for r in results])

print("=== CATEGORY DISTRIBUTION ===\n")
for category, count in categories.most_common():
    pct = count / len(results) * 100
    bar = "█" * int(pct / 2)
    print(f"{category:12} {count:4} ({pct:5.1f}%) {bar}")

print(f"\n📊 Total classified: {len(results)}")

# Cell 19: Save classification results
results_path = PROJECT / "data/parsed/classification_results.json"

with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(results)} results to {results_path}")